# CALIBRATION by Guill

In [1]:
import sys; sys.path.insert(0, '../'); from lib import *;
figure_features()

You don't have latex installed. Changing default configuration to tex=False


In [2]:
# Set options for general visualitation
OPT  = {
    "MICRO_SEC":   True,                # Time in microseconds (True/False)
    "NORM":        False,               # Runs can be displayed normalised (True/False)
    "ALIGN":       True,                # Aligns waveforms in peaktime (True/False)
    "LOGY":        False,               # Runs can be displayed in logy (True/False)
    "SHOW_AVE":    "",                  # If computed, vis will show average (AveWvf,AveWvfSPE,etc.)
    "SHOW_PARAM":  False,               # Print terminal information (True/False)
    "CHARGE_KEY":  "ChargeAveRange",    # Select charge info to be displayed. Default: "ChargeAveRange" (if computed)
    "PEAK_FINDER": False,               # Finds possible peaks in the window (True/False)
    "LEGEND":      True,                # Shows plot legend (True/False)
    "SHOW":        True
    }

In [32]:
INPUT_FILE = "MegaCell_LAr"; OV = 2 #To Select the run: OV=0 -> run=103; OV=1 -> run=113; OV=2 -> run=105; OV=3 -> run=111  
PRESET ="ALL"
info = read_input_file(INPUT_FILE)  # Read input file
channels = [0,1,6,7]

#-------------------------------- LOAD RUNS ---------------------------------#
run_keys = ["CALIB_RUNS","ALPHA_RUNS","MUONS_RUNS","NOISE_RUNS"]
nruns = dict.fromkeys(run_keys)
for key in run_keys:
    try:               nruns[key] = info[key][OV] # Store runs in dictionary
    except IndexError: nruns.pop(key)
print(nruns)

runs = dict.fromkeys(nruns.keys())
for run in runs: runs[run] = load_npy(np.asarray([nruns[run]]).astype(int),np.asarray(channels).astype(int),preset=PRESET,info=info,compressed=True)

{'CALIB_RUNS': 105, 'NOISE_RUNS': 106}
load_npy --> DONE!

load_npy --> DONE!

load_npy --> DONE!

load_npy --> DONE!

load_npy --> DONE!

load_npy --> DONE!

load_npy --> DONE!

load_npy --> DONE!



In [33]:
RUN2PLOT='CALIB_RUNS'
Charge_R_3={};
Charge_R_2={};
Charge_R_1={};
AnaADC={};
Lim_ped={};
Mean_AnaADC={};
MaxMin={};

for c in channels:
    events=np.arange(0,len(runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawADC'])-1)
    AnaADC[c]=[];
    for ev in events:
        AnaADC[c].append(runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawADC'][ev]-runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawPedMean'][ev]);

for c in channels:
    events=np.arange(0,len(runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawADC'])-1)
    Charge_R_1[c]=[];
    Charge_R_2[c]=[];
    Charge_R_3[c]=[];
    #AnaADC[c]=runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawADC']-runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawPedMean'];
    Mean_AnaADC[c]=np.mean(AnaADC[c],axis=0)
    Lim_ped[c]=runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawPedLim']
    if c == {6,7}:
        MaxMin[c]=np.argmin(Mean_AnaADC[c])
    else:
        MaxMin[c]=np.argmax(Mean_AnaADC[c])
    for ev in events:
        #AnaADC[c].append(runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawADC'][ev]-runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['RawPedMean'][ev]);
        #Ana_Int[c].append(AnaADC[c][ev][Lim_ped[c]:np.where(AnaADC[c][ev][Lim_ped:]<0)[0][0]])
        Charge_R_1[c].append(np.sum(AnaADC[c][ev][Lim_ped[c]:]*runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['Sampling']))
        Charge_R_2[c].append(np.sum(AnaADC[c][ev][Lim_ped[c]:(2*MaxMin[c]-Lim_ped[c])]*runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['Sampling']))
        Charge_R_3[c].append(np.sum(AnaADC[c][ev][np.int(1/2*(Lim_ped[c]+MaxMin[c])):(2*MaxMin[c]-Lim_ped[c])]*runs[RUN2PLOT][runs[RUN2PLOT]["NRun"][0]][c]['Sampling']))

/afs/ciemat.es/user/g/guilmaro/SCINT/.venv/lib/python3.7/site-packages/ipykernel_launcher.py:33: DeprecationWarning:

`np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations



# TRYING TO FIT

In [71]:
from scipy.stats import norm
#ch=np.random.choice(channels)
ch=0;
h, edges = np.histogram(Charge_R_3[ch],bins=4000);

def gaussian(x, mu, sigma, amplitude):
    return amplitude * norm.pdf(x,mu,sigma)

def sum_of_gaussians(x, *params):
    n_gaussians = len(params) // 3
    result = np.zeros_like(x)
    for i in range(n_gaussians):
        mu, sigma, amplitude = params[i * 3], params[i * 3 + 1], params[i * 3 + 2]
        result += gaussian(x, mu, sigma, amplitude)
    return result

# Define an initial guess for the parameters (mean, standard deviation, and amplitude) of the Gaussian components
initial_guess = [0, 0.5e-6, 1, 4e-6, 0.5e-6, 0.5,7e-6,2e-6,0.2]

params, _ = curve_fit(sum_of_gaussians, edges[:-1], h/np.max(h), p0=initial_guess)

adjusted_distribution = sum_of_gaussians(edges, *params)


In [72]:
run=105;
fig1=go.Figure();
fig1.add_trace(go.Bar(x=edges[:-1],y=h/np.max(h),name='Charge Histogram',opacity=1))
fig1.add_trace(go.Scatter(x=edges,y=adjusted_distribution,name='Fitted distribution'))
fig1.update_yaxes(title='Normalized counts',type='log',range=(-2,0))
fig1.update_xaxes(title='Charge (ADC counts * s)',range=(np.min(Charge_R_3[ch]),np.max(Charge_R_3[ch])))
fig1.update_layout(title='Channel = '+str(ch)+' Run = '+str(run),title_x=0.5,bargap = 0,template='simple_white')
fig1.show()

In [70]:
print(*params)

-1.2839261754033688e-08 8.323546335940545e-07 1.64234612194205e-06 3.3384450416721498e-06 6.69080342525363e-07 4.7471989891265233e-07 4.785526571988807e-06 3.7323041157222972e-06 9.72125743162937e-07
